In [ ]:
from typing import Literal, Optional, Tuple, Dict
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import DataLoader, random_split, Subset
from submodlib.functions.facilityLocation import FacilityLocationFunction

In [ ]:
# 加载 CIFAR-10 数据集
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
trainval_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_size = int(0.9 * len(trainval_dataset))
val_size = len(trainval_dataset) - train_size
train_dataset, val_dataset = random_split(trainval_dataset, [train_size, val_size])

# 将数据集转换为数组形式，方便使用
X_train = np.array([np.array(train_dataset[i][0]).flatten() for i in range(len(train_dataset))])
y_train = np.array([train_dataset[i][1] for i in range(len(train_dataset))])

In [ ]:
# 定义选择核心集合的函数
def select_coreset_class(B: int, c: int, X: np.ndarray, y: np.ndarray,
                         metric: Literal["euclidean", "cosine"] = "euclidean",
                         mode: Literal["dense", "sparse", "clustered"] = "dense", num_n: Optional[int] = None) \
        -> Tuple[np.ndarray, np.ndarray]:
    class_indices = np.where(y == c)[0]
    X = X[class_indices]
    N = X.shape[0]
    if N < B:
        raise ValueError("The size of the coreset for the given class must be smaller than or equal to the number of samples of the given class in the original dataset")
    if N == B:
        return class_indices, np.ones(N, dtype=np.float32)

    if mode == "dense":
        num_n = None

    obj = FacilityLocationFunction(n=len(X), mode=mode, data=X, metric=metric, num_neighbors=num_n)
    greedyList = obj.maximize(budget=B, optimizer="LazyGreedy", stopIfZeroGain=False, stopIfNegativeGain=False, verbose=False)
    order = list(map(lambda x: x[0], greedyList))
    order = np.asarray(order, dtype=np.int64)

    S = obj.sijs
    sz = np.zeros(B, dtype=np.float64)
    for i in range(N):
        if np.max(S[i, order]) <= 0:
            continue
        sz[np.argmax(S[i, order])] += 1
    sz[np.where(sz == 0)] = 1

    return class_indices[order], sz

In [ ]:
def select_coreset(B: int, X: np.ndarray, metric: Literal["euclidean", "cosine"] = "euclidean",
                   y: Optional[np.ndarray] = None, equal_num: bool = False,
                   mode: Literal["dense", "sparse", "clustered"] = "dense", num_n: Optional[int] = None) \
        -> Tuple[np.ndarray, np.ndarray, Dict[int, Tuple[np.ndarray, np.ndarray]]]:
    N = X.shape[0]
    if N < B:
        raise ValueError("The size of the coreset must be smaller than or equal to the size of the original dataset")
    if y is None:
        y = np.zeros(N, dtype=np.int32)
    classes = np.unique(y)
    classes = classes.astype(np.int32).tolist()
    C = len(classes)

    if equal_num:
        class_nums = np.asarray([sum(y == c) for c in classes], dtype=np.int32)
        num_per_class = int(np.ceil(B / C)) * np.ones(C, dtype=np.int32)
        minority = class_nums < np.ceil(B / C)
        if sum(minority) > 0:
            num_per_class[minority] = class_nums[minority]
            extra = sum([max(0, np.ceil(B / C) - class_nums[c]) for c in classes])
            minority = (class_nums - num_per_class) <= 0
            while extra > 0:
                index_not_minority = np.arange(C, dtype=np.int32)[~minority]
                index = index_not_minority[np.argmin(num_per_class[~minority])]
                num_per_class[index] += 1
                minority = (class_nums - num_per_class) <= 0
                extra -= 1
    else:
        num_per_class = np.int32(np.ceil(np.divide([sum(y == i) for i in classes], N) * B))
        diff = np.sum(num_per_class) - B
        if diff > 0:
            num_per_class -= np.concatenate((np.ones(diff, dtype=np.int32), np.zeros(C - diff, dtype=np.int32)))

    order_mg_all, cluster_sizes_all = zip(*map(lambda c: select_coreset_class(num_per_class[c[0]], c[1], X, y, metric, mode, num_n), enumerate(classes)))
    order_mg = np.concatenate(order_mg_all, dtype=np.int32)
    weights_mg = np.concatenate(cluster_sizes_all, dtype=np.float32)
    order_weights_classes = {c: (order_mg_all[ind], cluster_sizes_all[ind]) for ind, c in enumerate(classes)}

    return order_mg, weights_mg, order_weights_classes

In [ ]:
# 使用核心集合函数选择核心集合
B = 5000  # 假设选择 5000 个样本作为核心集合
selected_indices, weights, _ = select_coreset(B=B, X=X_train, y=y_train)

# 使用选择的核心集合构建新的训练集
coreset_dataset = Subset(train_dataset, selected_indices)

In [ ]:
# 定义简单的卷积神经网络模型
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = nn.functional.relu(self.conv1(x))
        x = nn.functional.max_pool2d(x, 2, 2)
        x = nn.functional.relu(self.conv2(x))
        x = nn.functional.max_pool2d(x, 2, 2)
        x = x.view(-1, 32 * 8 * 8)
        x = nn.functional.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
# 训练模型并评估
train_loader = DataLoader(coreset_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

device = torch.device("cpu")
print('device', device)

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# 训练过程
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    print(f"Epoch {epoch + 1}, Loss: {running_loss / len(train_loader)}")


In [ ]:
# 评估模型
model.eval()
y_true = []
y_pred = []
with torch.no_grad():
    for inputs, labels in val_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

# 计算评估指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='macro')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")